In [2]:
import pandas as pd 
import numpy as np 

dt = '20110107'
dt = pd.to_datetime(dt)

In [3]:
dt

Timestamp('2011-01-07 00:00:00')

In [4]:
pd.to_datetime('2011-01-07')

Timestamp('2011-01-07 00:00:00')

In [5]:
# get time fields
print (dt.year)

print (dt.month)

print (dt.day)

# maybe you care about price behavior on a particular hour, minute or second of the day?
print (dt.hour)

print (dt.minute)

print (dt.second)

2011
1
7
0
0
0


In [6]:
# useful for checking if weekend
dt.weekday()

4

In [7]:
# handles hours/minute/seconds as well
pd.to_datetime('20110107 12:10:30')

Timestamp('2011-01-07 12:10:30')

In [8]:
# convert back to a string
dt.strftime('%Y%m%d')

'20110107'

Check documentation for other string formatting codes: https://docs.python.org/3/library/datetime.html#strftime-and-strptime-behavior

# Pandas Time Series: Understanding `shift()` and Memory

## 🧠 Core Concept

`shift()` does **not** modify your DataFrame.

It creates a **temporary, aligned Series** that exists in memory and is used for calculations.

---

## 🔧 Example

    df['price'].shift(1)

- Returns a new Series
- Same index as original
- Values moved down (previous values aligned with current row)
- Not stored unless assigned

---

## 🔄 How It Works in a Calculation

    df['price'] / df['price'].shift(1)

### Step-by-step:

1. Original Series:

       100
       110
       105

2. Shifted Series (temporary):

       NaN
       100
       110

3. Pandas aligns them by index:

       price    shifted
       100      NaN
       110      100
       105      110

4. Performs element-wise calculation:

       100 / NaN
       110 / 100
       105 / 110

5. Returns a new Series:

       NaN
       1.10
       0.95

---

## ⚠️ Important Behavior

This does NOT change the DataFrame:

    returns = df['price'] / df['price'].shift(1)

- `df` remains unchanged
- `returns` is a standalone Series

---

## ✅ When the DataFrame Changes

Only when you explicitly assign:

    df['returns'] = df['price'] / df['price'].shift(1)

- Now a new column is added to `df`

---

## 🧠 Mental Model

Think of `shift()` as:

> Creating a temporary version of past data, aligned with the present

---

## 🔑 Key Rule

All Pandas operations:

- Return new objects
- Do NOT modify the original DataFrame unless assigned

---

## 🧭 Final Takeaway

`shift()`:

- Creates temporary, aligned historical data
- Enables comparisons across time
- Produces new results without modifying the original data unless explicitly assigned

In [9]:
# can shift dates
dt + pd.tseries.offsets.Day()

Timestamp('2011-01-08 00:00:00')

In [10]:
# can shift by multiple time units
dt + pd.tseries.offsets.Day(2)

Timestamp('2011-01-09 00:00:00')

In [11]:
# shift only by business day (skip weekends)
dt + pd.tseries.offsets.BDay()

Timestamp('2011-01-10 00:00:00')

In [12]:
# roll forward / backward
print (pd.tseries.offsets.MonthEnd().rollforward(dt))
print (pd.tseries.offsets.MonthEnd().rollback(dt))

2011-01-31 00:00:00
2010-12-31 00:00:00


Check documentation for other offsets: https://pandas.pydata.org/docs/reference/offset_frequency.html

In [13]:
# subtract two times 
# maybe for checking days till next earnings?

dt1 = pd.to_datetime('20110101')
dt2 = pd.to_datetime('20120630')

diff = dt2 - dt1
diff 

Timedelta('546 days 00:00:00')

In [14]:
diff.days

546

# Pandas `DatetimeIndex`: When and Why to Use It

## 🧠 Core Idea

`pd.DatetimeIndex()` converts date-like data into a **time-aware index** that Pandas can understand and operate on efficiently.

---

## 🔧 Example

    date = ['20110102','20110103','20110105']
    dt_index = pd.DatetimeIndex(date)

---

## 🚀 When You Use This

### 1. Converting Raw Date Strings

**Common scenario:**
- CSV / Excel files store dates as strings

Example:

    '20110102'  →  '2011-01-02'

Why:
- Enables time-based operations
- Prevents string-based errors

---

### 2. Setting a Time-Based Index

    df.index = pd.DatetimeIndex(df['date'])

Why:
- Allows Pandas to treat data as time series
- Unlocks time-based slicing and filtering

---

### 3. Time-Based Filtering

    df['2011-01-02':'2011-01-05']

Why:
- Easy selection by date ranges
- No manual filtering logic needed

---

### 4. Resampling Data

    df.resample('D').mean()

Why:
- Aggregate data to:
  - daily
  - weekly
  - monthly
- Critical for financial and engineering time series

---

### 5. Aligning Time Series

Why:
- Ensures multiple datasets line up by time
- Essential for:
  - multi-asset analysis
  - sensor data
  - event comparison

---

### 6. Feature Engineering (Time-Based)

    df.index.day
    df.index.month
    df.index.weekday

Why:
- Extract meaningful patterns:
  - seasonality
  - trends
  - cyclical behavior

---

### 7. Rolling and Time-Based Calculations

    df.rolling('5D').mean()

Why:
- Time-aware rolling windows (not just row-based)
- Useful for:
  - moving averages
  - smoothing noisy data

---

## ⚡ Why Not Just Keep Strings?

Because strings:
- cannot be sorted correctly in all formats
- cannot be used for time math
- cannot be resampled
- cannot be shifted properly in time context

---

## 🧭 Key Takeaway

Use `DatetimeIndex` when:

> You want Pandas to understand that your data represents **time**, not just text.

---

## 💡 Mental Model

- Strings = labels
- DatetimeIndex = **timeline**

Once data is on a timeline:
- filtering becomes natural
- calculations become meaningful
- analysis becomes scalable

In [15]:
# faster way to create multiple timestamp objects. useful for converting excel dates from strings to timestamps
date = ['20110102','20110103','20110105']
dt_index = pd.DatetimeIndex(date)
dt_index

DatetimeIndex(['2011-01-02', '2011-01-03', '2011-01-05'], dtype='datetime64[us]', freq=None)

In [16]:
dt_index[0]

Timestamp('2011-01-02 00:00:00')

In [17]:
## get time range 
days = pd.date_range(dt1,dt2,freq='D')
days

DatetimeIndex(['2011-01-01', '2011-01-02', '2011-01-03', '2011-01-04',
               '2011-01-05', '2011-01-06', '2011-01-07', '2011-01-08',
               '2011-01-09', '2011-01-10',
               ...
               '2012-06-21', '2012-06-22', '2012-06-23', '2012-06-24',
               '2012-06-25', '2012-06-26', '2012-06-27', '2012-06-28',
               '2012-06-29', '2012-06-30'],
              dtype='datetime64[us]', length=547, freq='D')

In [18]:
# can use strings as inputs and also use monthly frequency
months = pd.date_range('20110101','20121231',freq='ME')
months

DatetimeIndex(['2011-01-31', '2011-02-28', '2011-03-31', '2011-04-30',
               '2011-05-31', '2011-06-30', '2011-07-31', '2011-08-31',
               '2011-09-30', '2011-10-31', '2011-11-30', '2011-12-31',
               '2012-01-31', '2012-02-29', '2012-03-31', '2012-04-30',
               '2012-05-31', '2012-06-30', '2012-07-31', '2012-08-31',
               '2012-09-30', '2012-10-31', '2012-11-30', '2012-12-31'],
              dtype='datetime64[us]', freq='ME')

See documentation for other frequencies: https://pandas.pydata.org/docs/user_guide/timeseries.html#timeseries-offset-aliases

# Pandas Time Series: Why Create This Structure?

## 🧠 The Misconception

It may seem like this code is teaching you how to create fake data:

    df = pd.DataFrame(
        np.random.randn(len(days), len(univ)),
        index=days,
        columns=univ
    )

In reality, the random data is irrelevant.

---

## 🎯 The Real Purpose

This code demonstrates the **correct structure for time-series data** in Pandas.

Specifically:

- A **DatetimeIndex** (time as the index)
- Columns representing entities (e.g., assets, sensors, metrics)

---

## 🧱 The Structure That Matters

| Time (index) | SPY | TLT | VXX | QQQ |
|--------------|-----|-----|-----|-----|
| 2019-01-01 00:00 | x | x | x | x |
| 2019-01-01 00:01 | x | x | x | x |

This is the foundation for all time-based operations in Pandas.

---

## 🚀 Why This Structure Is Important

Once your data is in this format, you can:

### 1. Slice by Time

    df['2020-03-01':'2020-04-01']

→ Select specific time periods easily

---

### 2. Resample Data

    df.resample('D').mean()

→ Convert minute data to daily data

---

### 3. Aggregate Time Periods

    df.resample('M').last()

→ Get monthly closing values

---

### 4. Rolling Calculations

    df.rolling(60).mean()

→ Moving averages over time

---

### 5. Align Multiple Datasets

    df1 + df2

→ Automatically aligns on timestamps

---

## 🔄 Real-World Workflow

You will NOT generate random data.

You WILL do something like this:

    df = api.get_data(...)
    df.index = pd.to_datetime(df['timestamp'])

Possibly followed by reshaping:

    df = df.pivot(index='timestamp', columns='asset', values='price')

---

## 💡 Key Insight

The goal is not to create data.

The goal is to ensure your data is:

> **indexed by time and structured for time-based operations**

---

## 🧭 Final Takeaway

If your dataset is not in this format:

- Time as the index
- Columns as entities

Then many Pandas time-series features will not work properly.

---

## 🔑 Mental Model

- Raw data → messy, unstructured
- Proper DataFrame → structured timeline

Once your data becomes a timeline:
- analysis becomes easier
- transformations become natural
- operations scale cleanly

In [19]:
## DataFrames with pandas datetime index 
univ = ['SPY','TLT','VXX','QQQ']
days = pd.date_range('20190101','20210630',freq='min')
df = pd.DataFrame(np.random.randn(len(days),len(univ)),index=days,columns=univ)
df


,SPY,TLT,VXX,QQQ
2019-01-01 00:00:00,-0.387781,-0.223350,1.316990,-0.814298
2019-01-01 00:01:00,-0.246830,-0.016378,-0.730045,-0.468184
2019-01-01 00:02:00,-1.424965,-1.883197,0.356314,1.324045
2019-01-01 00:03:00,-0.231921,0.859517,-1.899798,-1.113192
2019-01-01 00:04:00,-2.176040,1.094873,0.455876,-0.421660
...,...,...,...,...
2021-06-29 23:56:00,-0.107576,-0.292173,-1.848287,0.682322
2021-06-29 23:57:00,0.396228,-0.614806,-0.246388,0.476845
2021-06-29 23:58:00,0.033187,-0.565288,-0.748721,0.832706
2021-06-29 23:59:00,-0.613235,1.437189,0.486089,-0.575148


In [20]:
df.loc['2019-01-01 00:00:00']

SPY   -0.387781
TLT   -0.223350
VXX    1.316990
QQQ   -0.814298
Name: 2019-01-01 00:00:00, dtype: float64

In [21]:
df.loc['2019-01-01']

,SPY,TLT,VXX,QQQ
2019-01-01 00:00:00,-0.387781,-0.223350,1.316990,-0.814298
2019-01-01 00:01:00,-0.246830,-0.016378,-0.730045,-0.468184
2019-01-01 00:02:00,-1.424965,-1.883197,0.356314,1.324045
2019-01-01 00:03:00,-0.231921,0.859517,-1.899798,-1.113192
2019-01-01 00:04:00,-2.176040,1.094873,0.455876,-0.421660
...,...,...,...,...
2019-01-01 23:55:00,-0.850374,1.636809,-0.896068,0.652587
2019-01-01 23:56:00,0.704829,-1.758781,0.330345,0.539126
2019-01-01 23:57:00,-1.632930,0.186968,0.026877,0.921289
2019-01-01 23:58:00,1.699863,-0.267897,0.553256,-1.285359


In [23]:
df.resample('ME').sum()

,SPY,TLT,VXX,QQQ
2019-01-31,92.101076,-167.818412,27.776907,151.731144
2019-02-28,-133.606867,0.337018,-71.252041,58.363138
2019-03-31,122.732559,28.077957,-116.620583,-172.154814
2019-04-30,78.181170,137.978733,103.397687,75.768399
2019-05-31,-321.195057,12.531633,-546.773263,124.164545
2019-06-30,130.029271,16.795884,-68.701902,41.313879
2019-07-31,-206.416833,22.810437,-468.258668,-122.344834
2019-08-31,18.979504,-101.432224,169.119241,80.848687
2019-09-30,158.008529,-130.917376,-33.478807,-63.588005
2019-10-31,-78.943750,-355.062579,147.560793,434.885181


In [24]:
# 5 minute high 
df.resample('5min').max()

,SPY,TLT,VXX,QQQ
2019-01-01 00:00:00,-0.231921,1.094873,1.316990,1.324045
2019-01-01 00:05:00,0.786928,1.749326,1.510614,0.850069
2019-01-01 00:10:00,1.370599,2.351350,0.772986,1.748437
2019-01-01 00:15:00,0.804894,1.229790,1.118933,0.501924
2019-01-01 00:20:00,1.771905,1.600962,0.584103,1.143193
...,...,...,...,...
2021-06-29 23:40:00,0.066128,0.289935,1.231946,0.552187
2021-06-29 23:45:00,1.297894,0.132472,2.099928,1.584746
2021-06-29 23:50:00,0.931905,0.619315,1.442119,1.477923
2021-06-29 23:55:00,1.047938,1.437189,0.486089,1.391744


In [25]:
# close
df.resample('5min').last()

,SPY,TLT,VXX,QQQ
2019-01-01 00:00:00,-2.176040,1.094873,0.455876,-0.421660
2019-01-01 00:05:00,0.359604,0.944727,-0.354089,-0.425754
2019-01-01 00:10:00,-0.743472,-1.141125,-0.157878,0.112158
2019-01-01 00:15:00,0.804894,1.229790,0.519964,-0.990418
2019-01-01 00:20:00,1.274663,1.600962,-0.898899,0.418395
...,...,...,...,...
2021-06-29 23:40:00,-0.095522,-0.827625,1.231946,0.464361
2021-06-29 23:45:00,-0.703822,-0.974788,0.341309,0.433805
2021-06-29 23:50:00,0.931905,-0.937766,0.324011,-0.869929
2021-06-29 23:55:00,-0.613235,1.437189,0.486089,-0.575148


Again, refer to documentation for how to specify the frequencies: https://pandas.pydata.org/docs/user_guide/timeseries.html#timeseries-offset-aliases


Many other topics on time series in pandas not covered here!